# CHAPTER 2 - The Tools of the Trade in Quantum Computing - PennyLane Code

*Note*: You may skip the following three cells if you have alredy installed the right versions of all the libraries mentioned in *Appendix D*. This will likely NOT be the case if you are running this notebook on a cloud service such as Google Colab.



In [1]:
import pennylane as qml

In [2]:
dev = qml.device('default.qubit', wires = 2)

In [3]:
def qc():
    qml.PauliX(wires = 0)
    qml.Hadamard(wires = 0)
    return qml.state()

In [4]:
qcirc = qml.QNode(qc, dev) # Assemble the circuit & the device.
qcirc() # Run it!

array([ 0.70710678+0.j,  0.        +0.j, -0.70710678+0.j,  0.        +0.j])

In [5]:
@qml.qnode(dev) # We add this decorator to use the device dev.
def qcirc():
    qml.PauliX(wires = 0)
    qml.Hadamard(wires = 0)
    return qml.state()
    
# Now qcirc is already a QNode. We can just run it!
qcirc()

array([ 0.70710678+0.j,  0.        +0.j, -0.70710678+0.j,  0.        +0.j])

In [6]:
dev = qml.device('default.qubit', wires = 1)
@qml.qnode(dev)
def qcirc(theta):
    qml.RX(theta, wires = 0)
    return qml.state()

In [7]:
print(qml.draw(qcirc)(theta = 2))

0: ──RX(2.00)─┤  State


In [8]:
dev = qml.device('default.qubit', wires = 3)

# Get probabilities
@qml.qnode(dev)
def qcirc():
    qml.Hadamard(wires = 1)
    return qml.probs(wires = [1, 2]) # Only the last 2 wires.
prob = qcirc()
print("Probs. wires [1, 2] with H in wire 1:", prob)

# Get a sample, not having specified shots in the device.
@qml.qnode(dev)
def qcirc():
    qml.Hadamard(wires = 0)
    return qml.sample(wires = 0) # Only the first wire.
s1 = qcirc(shots = 4) # We specify the shots here.
print("Sample 1 after H:", s1)

# Get a sample with shots in the device.
dev = qml.device('default.qubit', wires = 2, shots = 4)
@qml.qnode(dev)
def qcirc():
    qml.Hadamard(wires=0)
    return qml.sample() # Will sample all wires.
s2 = qcirc()
print("Sample 2 after H x I:", s2)

Probs. wires [1, 2] with H in wire 1: [0.5 0.  0.5 0. ]
Sample 1 after H: [0 1 0 1]
Sample 2 after H x I: [[0 0]
 [0 0]
 [1 0]
 [0 0]]


In [9]:
dev = qml.device('qiskit.aer', wires = 2)
@qml.qnode(dev)
def qcirc():
    qml.Hadamard(wires = 0)
    return qml.probs(wires = 0)
s = qcirc()
print("The probabilities are", s)

The probabilities are [0.49609375 0.50390625]


*Note*: In the following cell, you need to replace "1234" with your actual IBM token. Refer to *Appendix D* in the book for instructions on how to create an account and get your token. Be very careful not to disclose your token to anyone!

### Run on IBM quantum device

In [10]:
from qiskit_ibm_runtime import QiskitRuntimeService

In [11]:
service = QiskitRuntimeService()
bck = service.backend("ibm_brisbane")

/tmp/ipykernel_35221/3129341905.py:1: DeprecationWarning: The "ibm_quantum" channel option is deprecated and will be sunset on 1 July. After this date, "ibm_cloud", "ibm_quantum_platform", and "local" will be the only valid channels. Open Plan users should migrate now.  All other users should review the migration guide (https://quantum.cloud.ibm.com/docs/migration-guides/classic-iqp-to-cloud-iqp)to learn when to migrate.
  service = QiskitRuntimeService()


In [19]:
# Invoke the PennyLane IBM quantum device.
real_dev = qml.device('qiskit.remote', wires = 5, backend = bck, shots=1024)

In [24]:
# Send a circuit and get some results!
@qml.qnode(real_dev)
def real_qcirc():
    qml.Hadamard(wires = 0)
    return qml.probs(wires = 0)

#s = real_qcirc()
#print("The probabilities are", s)

In [23]:
# Pennylane doc example
@qml.qnode(real_dev)
def circuit(x, y, z):
    qml.RZ(z, wires=[0])
    qml.RY(y, wires=[0])
    qml.RX(x, wires=[0])
    qml.CNOT(wires=[0, 1])
    return qml.expval(qml.PauliZ(wires=1))

In [22]:
# Runs with 1024 shots
# circuit(0.2, 0.1, 0.3)